## RAG with PDF Data extraction to give context to LLM

In [1]:
from dotenv import load_dotenv
import os

load = load_dotenv('./../.env', override=True)

#print(os.getenv('LANGSMITH_PROJECT'))

In [ ]:
from langchain_ollama import ChatOllama
import os

ollama_cloud_llm = ChatOllama(
    base_url="http://localhost:11434/",  # Ollama cloud endpoint
    model="devstral-small-2:24b-cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
    temperature=0.5,
    max_tokens=450,
    headers={
        "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
    }
)

ollama_local_llm = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest",
    temperature=0.5,
    max_tokens=500,
    num_gpu=999
)

### 1. Extraction the PDF files

In [51]:
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "./attention.pdf"
pdf2 = "./LLMForgetting.pdf"
pdf3 = "./TestingAndEvaluatingLLM.pdf"

pdfFiles = [pdf1, pdf2, pdf3]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    documents.extend(docs)

print(f"Total documents loaded: {len(documents)}")

Total documents loaded: 253


In [53]:
print(f"Get document content by index 0: {documents[0]}")

Get document content by index 0: page_content='Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensin

### 2. Text Splitting

In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_splits = text_splitter.split_documents(documents)

len(all_splits)

640

### 3. Embeddings

In [56]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="llama3.2:latest")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)

print(f"Length of embedding vector: {len(vector_1)}")

Length of embedding vector: 3072


### 4. Vector Stores

In [57]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db" #remove if not necessary
)

### 5. Retriving from the Persistant Vector DB

In [58]:
vector_store = Chroma(persist_directory='./chroma_langchain_db', embedding_function=embeddings)

result = vector_store.similarity_search("What is Bias testing", k=3)

result

[Document(id='0869d2e9-5dc5-443f-b71a-b48ff4c28a69', metadata={'page': 78, 'total_pages': 223, 'title': '', 'author': '', 'trapped': '/False', 'creationdate': '2024-09-04T00:37:21+00:00', 'creator': 'LaTeX with hyperref', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'start_index': 0, 'keywords': '', 'producer': 'pdfTeX-1.40.25', 'page_label': '60', 'subject': '', 'source': './TestingAndEvaluatingLLM.pdf', 'moddate': '2024-09-04T00:37:21+00:00'}, page_content='60 CHAPTER 4. LOGICAL REASONING CORRECTNESS\nthe following challenges: 1) If an LLM concludes correctly, it is unclear\nwhether the response stems from reasoning or merely relies on simple\nheuristics such as memorization or word correlations (e.g., “dry floor”\nis more likely to correlate with “playing football”). 2) If an LLM\nfails to reason correctly, it is not clear which part of the reasoning\nprocess it failed (i.e., inferring not raining from floor being dry o

In [59]:
result = vector_store.similarity_search_with_score("What are the types of LLM testing?")

result[0]

(Document(id='00f53155-e607-42c7-b412-8d6426dedd37', metadata={'source': './TestingAndEvaluatingLLM.pdf', 'title': '', 'author': '', 'page': 39, 'creator': 'LaTeX with hyperref', 'keywords': '', 'page_label': '21', 'total_pages': 223, 'creationdate': '2024-09-04T00:37:21+00:00', 'start_index': 0, 'producer': 'pdfTeX-1.40.25', 'subject': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'trapped': '/False', 'moddate': '2024-09-04T00:37:21+00:00'}, page_content='2.2. SOFTW ARE TESTING 21\nFigure 2.7: A diagram illustrating the three steps of our method: (1) supervised\nfine-tuning (SFT), (2) reward model (RM) training, and (3) reinforcement learning\nvia proximal policy optimization (PPO)on this reward model.\nshow promising generalization to instructions outside of the RLHF\nfinetuning distribution.\nLLAMA-2\nIn addition to the API-based LLMs, there is also a branch of\nopen-sourced LLMs. Llama-2 is a family of pretrained an

### 6. Retrivers in Langchain

In [60]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

retriever.batch(
    [
        "What is the Bias Mearsurements",
        "How to test human safety against LLM",
        "How LLM forgets the context"
    ]
)

[[Document(id='8a54646e-5a59-4912-bd13-11a4ab4d3ef2', metadata={'start_index': 0, 'trapped': '/False', 'total_pages': 15, 'keywords': '', 'page_label': '10', 'producer': 'pdfTeX-1.40.25', 'subject': '', 'author': '', 'moddate': '2025-01-07T01:36:50+00:00', 'creator': 'LaTeX with hyperref', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': './LLMForgetting.pdf', 'page': 9, 'creationdate': '2025-01-07T01:36:50+00:00', 'title': ''}, page_content='Under review\nFigure 6: The performance of general knowledge of the BLOOMZ-7.1b and LLAMA-7b\nmodel trained on the instruction data and the mixed data. The dashed lines refers to the\nperformance of BLOOMZ-7.1b and LLAMA-7B and the solid ones refer to those of mixed-\ninstruction trained models.\nincreases to 3b, BLOOMZ-3b suffers less forgetting compared to mT0-3.7B. For example, the\nFG value of BLOOMZ-3b is 11.09 which is 5.64 lower than that of mT0-3.7b. These results\nsugge

### Document Retrival Manually

In [65]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

query = "Explain how position-wise Feed-Forward network calculation works"
# query = "What exactly does Testing the Factual Correctness of LLM tells"
# query = "How to test translation in LLM?"

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# print(context_text)

prompt_template = ChatPromptTemplate.from_template(
    """ You are an AI Assistant. Use the following context to answer the question correctly. If you dont know the answer, just tell I dont know.
        "context: {context} \n\n"
        "question: {question} \n\n"
    AI answer:"""
)

# print(prompt_template)

chain = prompt_template | ollama_local_llm | StrOutputParser()

response = chain.invoke(
    {"context": context_text, "question": query}
)

print(response)

I don't know. The provided text doesn't mention anything about neural networks or the calculation of a position-wise Feed-Forward network. It appears to be related to bias detection in conversational AI systems, but it doesn't cover this specific topic. If you could provide more context or clarify what you're looking for, I'd be happy to try and help further!


### Using Langchain Hub for prompt

In [70]:
from langchain_core.output_parsers import StrOutputParser
from langchainhub import Client
import json
from langchain_core.load import load

query = "Explain how position-wise Feed-Forward network calculation works"
# query = "What exactly does Testing the Factual Correctness of LLM tells"
# query = "How to test translation in LLM?"

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

# prompt_template = ChatPromptTemplate.from_template(
#     """ You are an AI Assistant. Use the following context to answer the question correctly. If you dont know the answer, just tell I dont know.
#         "context: {context} \n\n"
#         "question: {question} \n\n"
#     AI answer:"""
# )

hub = Client()
prompt_data = hub.pull("rlm/rag-prompt")

prompt_dict = json.loads(prompt_data)

prompt = load(prompt_dict)

chain = prompt | ollama_local_llm | StrOutputParser()

response = chain.invoke(
    {"context": context_text, "question": query}
)

print(response)

/tmp/ipykernel_5979/1221428432.py:22: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt_data = hub.pull("rlm/rag-prompt")
/home/shashank-sharma/Developer/Projects/langchain-trainings/myenv312/lib/python3.14/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)
/tmp/ipykernel_5979/1221428432.py:26: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  prompt = load(prompt_dict)
/tmp/ipykernel_5979/1221428432.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppre

I don't know how a position-wise Feed-Forward network calculation works. I also don't see any information about it in the provided context. The text appears to be related to bias detection in conversational AI systems, but doesn't mention neural networks or their calculations.


### Retrieving data using RetrievalQA is obsolete. Use Chains instead.

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(ollama_local_llm, retriever = retriever, return_source_documents=True)

# question = "What is Training data and Batching"

# question = "Explain how position-wise Feed-Forward network calculation works"
query = "What exactly does Testing the Factual Correctness of LLM tells"

response = qa_chain.invoke(question)

sources = set(doc.metadata.get("source", "Unknown") for doc in response["spurce_documents"])

print(response['result'])
print("\n Sources used:")
for source in sources:
    print(f"- {source}")

ImportError: cannot import name 'RetrievalQA' from 'langchain_protocol' (/home/shashank-sharma/Developer/Projects/langchain-trainings/myenv312/lib/python3.14/site-packages/langchain_protocol/__init__.py)